# DS 208 · Week 11 — Reproducible workflows
## Git, environments and debugging

Every code block from the lecture, in order, so you can run what you watched instead of
retyping it. The first slides use small tables typed into the cells; the **Case study**
cell loads the flood control data from the course API, so there is no file to download.

**How to use this.** Run the cells top to bottom the first time (Runtime → Run all). The
exercise cells list their tasks above the code: predict first, then run, then change one
thing and run again.

**Version differences.** The slides follow pandas 3. On pandas 2, text columns show as
`object` instead of `str`, and dates as `datetime64[ns]` instead of `datetime64[us]`.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

---
## The lecture, cell by cell

Headings carry the slide numbers shown in the deck's counter.


### Slide 8 — Which blocks run this week


Expected output:

```
pandas version: 3.0.5
64-bit Python: True
city      str
pop     int64
dtype: object
```


In [ ]:
import io, sys

print("pandas version:", pd.__version__)
print("64-bit Python:", sys.maxsize > 2**32)

t = pd.read_csv(io.StringIO("city,pop\nCebu,964000"))
print(t.dtypes)

### Slide 35 — Week 7’s eight cities, again


Expected output:

```
(8, 3) (8, 2)
```


In [ ]:
import io

csv_text = """city,region,pop
Quezon City,Luzon,2960000
Manila,Luzon,1847000
Baguio,Luzon,366000
Cebu,Visayas,964000
Iloilo,Visayas,457000
Tacloban,Visayas,251000
Davao,Mindanao,1777000
Cagayan de Oro,Mindanao,728000"""
cities = pd.read_csv(io.StringIO(csv_text))
areas = pd.DataFrame({
    "city": ["Quezon City", "Manila", "Baguio", "Cebu",
             "Iloilo", "Tacloban", "Davao", "Cagayan de Oro"],
    "area_km2": [172, 42, 58, 315, 78, 202, 2444, 413]})

print(cities.shape, areas.shape)

### Slide 36 — The traceback states where and what


Expected output:

```
KeyError: 'popn'
```


This cell **fails on purpose**. It is wrapped in `try`/`except` so the notebook keeps running; the error is printed instead.


In [ ]:
try:
    # fails on purpose: read the last line
    # cities: the eight-city table
    print(cities["popn"].mean())      # a typo in the column name
except Exception as e:
    print(type(e).__name__ + ":", e)

### Slide 37 — Printing values and pausing to inspect


Expected output:

```
shape: (8, 3)
['city', 'region', 'pop']
```


In [ ]:
# cities: the eight-city table
print(f"shape: {cities.shape}")
print(cities.columns.tolist())

# breakpoint() here would pause and let you look (next slide)

### Slide 40 — Halving the suspect region to a single line


**Exercise.**

1. Predict after which step the row count changes.
2. Run the cell.
3. Fix it: in merge_regions, merge with regions.drop_duplicates(). What is the last count now?


In [ ]:
# cities: the eight-city table
regions = pd.DataFrame({"region": ["Luzon", "Luzon", "Visayas", "Mindanao"],
                        "island": ["Luzon", "Luzon", "Visayas", "Mindanao"]})

def load():             return cities.copy()
def clean(d):           return d.dropna()
def derive(d):          return d.assign(pop_k=d["pop"] / 1000)
def merge_regions(d):   return d.merge(regions, on="region", how="left")

df = load();             print(len(df))
df = clean(df);          print(len(df))
df = derive(df);         print(len(df))
df = merge_regions(df);  print(len(df))

### Slide 41 — The flood data, for the assert example


Expected output:

```
(34079, 16)
```


In [ ]:
import io, requests

url = "https://portal.latarak.com/api/flood/export.csv"
df  = pd.read_csv(io.StringIO(requests.get(url, timeout=60).text))

for col in ["start_date", "completion_date"]:
    df[col] = pd.to_datetime(df[col], errors="coerce")
df["duration"] = (df.completion_date - df.start_date).dt.days
print(df.shape)

### Slide 42 — Failing at the cause rather than three steps later


Expected output:

```
AssertionError: negative duration
```


This cell **fails on purpose**. It is wrapped in `try`/`except` so the notebook keeps running; the error is printed instead.


In [ ]:
try:
    # fails on purpose: read the last line
    regions = df[["region"]].drop_duplicates()    # one row per region
    before = len(df)
    df = df.merge(regions, on="region", how="left")
    assert len(df) == before, f"row count changed: {before} -> {len(df)}"

    assert df.duration.min() >= 0, "negative duration"
except Exception as e:
    print(type(e).__name__ + ":", e)

### Slide 43 — Log levels silence messages without deleting them


**Exercise.**

1. Predict which of the three messages print.
2. Run the cell.
3. Change level=logging.INFO to level=logging.WARNING. Which messages remain?


In [ ]:
# cities: the eight-city table
import logging, sys
logging.basicConfig(level=logging.INFO, stream=sys.stdout,
                    format="%(levelname)s %(message)s", force=True)
log = logging.getLogger("cities")

log.debug("columns: %s", cities.columns.tolist())
log.info("loaded %s cities", len(cities))
log.warning("%s cities have no area yet", len(cities))

### Slide 48 — Resolving a KeyError in three lines


**Exercise.**

1. Predict the error message.
2. Run the cell.
3. Change the fixed line to the median. What is the median population?


In [ ]:
# cities: the eight-city table
try:
    print(cities["popn"].mean())
except KeyError as e:
    print("KeyError:", e)

print(cities.columns.tolist())     # read the real names
print(cities["pop"].mean())        # fixed

---

*Data: DPWH flood control projects 2016–2026, mirrored by BetterGov.ph from the DPWH
transparency portal, CC0-1.0 (public domain). Served by the course API at* `/api/flood`.
